## Langfuse 프롬프트 관리 

### 1. Langfuse 계정 및 프로젝트 설정
1. [Langfuse Cloud](https://cloud.langfuse.com) 또는 Self-hosted 인스턴스에 가입
2. 프로젝트 생성 후 Settings에서 API 키 발급

### 2. 환경 변수 설정
`.env` 파일에 다음 내용을 추가하세요:
```
LANGFUSE_SECRET_KEY="sk-lf-..."
LANGFUSE_PUBLIC_KEY="pk-lf-..."
LANGFUSE_HOST="https://cloud.langfuse.com"
OPENAI_API_KEY="sk-..."
```

### 3. 필수 패키지 설치
```bash
uv add langfuse langchain-openai langchain-core python-dotenv
```

---

## 환경 설정 및 준비

### (1) Env 환경변수

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

### (2) 기본 라이브러리

In [2]:
import os
from glob import glob
from pprint import pprint
import json
import warnings
warnings.filterwarnings("ignore")

### (3) Langfuse 콜백 핸들러 설정

In [3]:
from langfuse.langchain import CallbackHandler 

# LangChain 콜백 핸들러 생성
langfuse_handler = CallbackHandler()

### (4) Langfuse 클라이언트 설정

In [4]:
from langfuse import Langfuse

# Langfuse 클라이언트 초기화
langfuse = Langfuse()

# 연결 테스트
assert langfuse.auth_check()

---

## 프롬프트 관리 개요

Langfuse는 **프롬프트 CMS(Content Management System)** 기능을 제공

- **버전 관리**: 프롬프트의 모든 변경사항을 추적하고 롤백 가능
- **협업**: 팀원들과 함께 프롬프트를 편집하고 관리
- **배포 관리**: 라벨을 통해 코드 변경 없이 환경별 배포
- **성능 모니터링**: 프롬프트 버전별 성능 메트릭 비교
- **실시간 테스트**: 플레이그라운드에서 즉시 테스트 가능

- 라벨(Labels) 이해
    - `production`: 프로덕션 환경에서 사용 중인 안정화된 버전
    - `staging`: 테스트 환경에서 검증 중인 버전
    - `latest`: 가장 최근에 생성된 버전 (자동 부여)
    - 사용자 정의 라벨: 자유롭게 지정 가능 (`v2-stable`, `experiment-a` 등)

---

## 1. 프롬프트 생성

### 1.1 텍스트 프롬프트 생성

In [5]:
# 텍스트 프롬프트 생성
langfuse.create_prompt(
    name="movie-critic",  # 프롬프트 이름
    type="text",          
    prompt="{{criticLevel}} 영화 평론가로서, {{movie}}를 어떻게 생각하시나요?",
    labels=["production"],             # 프로덕션 레이블
    tags=["movie", "qa", "text"],      # 태그
    config={
        "model": "gpt-4.1-mini",       # 사용할 LLM 모델명
        "temperature": 0.7,            # 응답의 창의성 (0.0~2.0)
        "max_tokens": 500              # 최대 생성 토큰 수
    }
)

### 1.2 챗 프롬프트 생성

In [10]:
# 챗 프롬프트 생성
langfuse.create_prompt(
    name="movie-critic-chat",  # 프롬프트 이름
    type="chat",          
    prompt=[
        {
            "role": "system",
            "content": "당신은 {{criticLevel}} 영화 평론가입니다."
        },
        {
            "role": "user",
            "content": "영화 {{movie}}를 어떻게 생각하시나요?"
        }
    ],
    labels=["production"],       # 프로덕션 레이블
    tags=["movie", "qa", "chat"],    # 태그
    config={
        "model": "gpt-4.1-mini",      # 사용할 LLM 모델명
        "temperature": 0.7,            # 응답의 창의성 (0.0~2.0)
        "max_tokens": 500              # 최대 생성 토큰 수
    }
)

### 1.3 메시지 플레이스홀더가 있는 챗 프롬프트

In [11]:
# 메시지 플레이스홀더를 포함한 챗 프롬프트
langfuse.create_prompt(
    name="movie-critic-with-history",
    type="chat",
    prompt=[
        {
            "role": "system",
            "content": "당신은 {{criticLevel}} 영화 평론가입니다."
        },
        {
            "type": "placeholder",
            "name": "chat_history"  # 대화 히스토리 삽입 지점
        },
        {
            "role": "user",
            "content": "영화 {{movie}}에 대해 어떻게 생각하시나요?"
        }
    ],
    labels=["production"],
    tags=["movie", "qa", "chat", "history"],
    config={
        "model": "gpt-4.1-mini",      # 사용할 LLM 모델명
        "temperature": 0.7,            # 응답의 창의성 (0.0~2.0)
        "max_tokens": 500              # 최대 생성 토큰 수
    }
)

### **[실습 1]**
텍스트 기반 프롬프트와 chat 기반 프롬프트를 각각 구현하고, Langfuse UI에서 확인하세요.

**단계별 가이드:**
1. `langfuse.create_prompt()`를 사용하여 텍스트 프롬프트 생성
2. `type="text"`로 지정하고, 적절한 변수(`{{변수명}}`)를 포함한 프롬프트 작성
3. `type="chat"`으로 챗 프롬프트 생성 (system, user role 구분)
4. `config`에 모델명, temperature, max_tokens 설정
5. Langfuse UI의 Prompts 탭에서 생성된 프롬프트 확인

**힌트:** 
- 텍스트 프롬프트는 단순 문자열 형태
- 챗 프롬프트는 role과 content를 가진 딕셔너리 리스트 형태

In [ ]:
# 텍스트 프롬프트 생성
# 여기에 코드를 작성하세요

In [ ]:
# 챗 프롬프트 생성
# 여기에 코드를 작성하세요

---

## 2. 프롬프트 활용

### 2.1 기본 프롬프트 가져오기

In [12]:
# 프로덕션 버전 가져오기
prompt = langfuse.get_prompt("movie-critic")

# 프롬프트 정보 출력
print(f"모델: {prompt.config['model']}")
print(f"온도: {prompt.config['temperature']}")
print(f"라벨: {prompt.labels}")
print(f"태그: {prompt.tags}")
print(f"프롬프트: {prompt.prompt}")
print("-" * 100)

# 랭체인 호환 프롬프트 출력
print(prompt.get_langchain_prompt())

모델: gpt-4.1-mini
온도: 0.7
라벨: ['production', 'latest']
태그: ['movie', 'qa', 'text']
프롬프트: {{criticLevel}} 영화 평론가로서, {{movie}}를 어떻게 생각하시나요?
----------------------------------------------------------------------------------------------------
{criticLevel} 영화 평론가로서, {movie}를 어떻게 생각하시나요?


### 2.2 compile 메서드 사용

- compile 메서드로 변수 삽입

In [13]:
# compile 메서드로 변수 삽입
compiled_prompt = prompt.compile(criticLevel="전문가", movie="인셉션")
print(compiled_prompt)

전문가 영화 평론가로서, 인셉션를 어떻게 생각하시나요?


### 2.3 챗 프롬프트 가져오기 및 컴파일

In [14]:
# 챗 프롬프트 가져오기
chat_prompt = langfuse.get_prompt("movie-critic-chat", type="chat")

# 챗 프롬프트 정보 출력
print(f"모델: {chat_prompt.config['model']}")
print(f"온도: {chat_prompt.config['temperature']}")
print(f"라벨: {chat_prompt.labels}")
print(f"태그: {chat_prompt.tags}")
print(f"프롬프트: {chat_prompt.prompt}")
print("-" * 100)

# 랭체인 호환 프롬프트 출력
print(chat_prompt.get_langchain_prompt())

모델: gpt-4.1-mini
온도: 0.7
라벨: ['production']
태그: ['movie', 'qa', 'chat']
프롬프트: [{'type': 'message', 'role': 'system', 'content': '당신은 {{criticLevel}} 영화 평론가입니다.'}, {'type': 'message', 'role': 'user', 'content': '영화 {{movie}}를 어떻게 생각하시나요?'}]
----------------------------------------------------------------------------------------------------
[('system', '당신은 {criticLevel} 영화 평론가입니다.'), ('user', '영화 {movie}를 어떻게 생각하시나요?')]


In [15]:
# 챗 프롬프트 컴파일
compiled_chat_prompt = chat_prompt.compile(criticLevel="전문가", movie="인셉션")
print(compiled_chat_prompt)

[{'role': 'system', 'content': '당신은 전문가 영화 평론가입니다.'}, {'role': 'user', 'content': '영화 인셉션를 어떻게 생각하시나요?'}]


### 2.4 메시지 플레이스홀더 활용

In [16]:
# 플레이스홀더가 있는 챗 프롬프트 가져오기
prompt_with_history = langfuse.get_prompt("movie-critic-with-history", type="chat")

# 대화 히스토리 정의
chat_history = [
    {"role": "user", "content": "안녕하세요!"},
    {"role": "assistant", "content": "안녕하세요! 영화에 대해 이야기해볼까요?"}
]

# 변수와 플레이스홀더를 모두 컴파일
compiled_with_history = prompt_with_history.compile(
            criticLevel="전문가",
            movie="인셉션", 
            chat_history=chat_history
        )

for message in compiled_with_history:
    print(message)
    print("-" * 20)

{'role': 'system', 'content': '당신은 전문가 영화 평론가입니다.'}
--------------------
{'role': 'user', 'content': '안녕하세요!'}
--------------------
{'role': 'assistant', 'content': '안녕하세요! 영화에 대해 이야기해볼까요?'}
--------------------
{'role': 'user', 'content': '영화 인셉션에 대해 어떻게 생각하시나요?'}
--------------------


### **[실습 2]**
"movie-critic-chat" 프롬프트를 Langfuse에서 가져와서 내용을 출력하고, compile 메서드를 사용해 변수에 적절한 값을 추가해보세요.

**단계별 가이드:**
1. `langfuse.get_prompt()`로 "movie-critic-chat" 프롬프트 가져오기
2. `type="chat"` 파라미터 지정
3. 프롬프트 정보 출력 (config, labels, prompt 등)
4. `compile()` 메서드로 `criticLevel`과 `movie` 변수에 값 할당
5. 컴파일된 메시지 출력

**힌트:**
- 챗 프롬프트는 메시지 리스트로 반환됨
- compile 결과는 for 문으로 순회하여 출력

In [ ]:
# chat 프롬프트 가져오기 및 컴파일
# 여기에 코드를 작성하세요

---

## 3. 프롬프트 버전 관리

### 3.1 새로운 버전 생성

In [17]:
# 새로운 버전 생성 (같은 이름 사용)
langfuse.create_prompt(
    name="movie-critic",  # 같은 이름 사용
    type="text",          
    prompt="당신은 {{criticLevel}} 영화 평론가입니다.\n\n영화 {{movie}}에 대한 상세한 분석을 제공해주세요. 연출, 연기, 스토리, 시각적 효과를 포함하여 평가해주세요.",
    labels=["production"],       # 프로덕션 레이블
    tags=["movie", "qa", "text", "detailed"],    # 태그 업데이트
    config={
        "model": "gpt-4.1",  # 모델 업그레이드
        "temperature": 0.7,
        "max_tokens": 1000  # 토큰 수 증가
    }
)

### 3.2 특정 버전 가져오기

In [18]:
# 특정 버전 가져오기
prompt_v1 = langfuse.get_prompt("movie-critic", version=1)
prompt_v2 = langfuse.get_prompt("movie-critic", version=2)

# 버전별 비교
print(f"V1 프롬프트: {prompt_v1.prompt}")
print(f"V2 프롬프트: {prompt_v2.prompt}")
print(f"V1 모델: {prompt_v1.config['model']}")
print(f"V2 모델: {prompt_v2.config['model']}")

V1 프롬프트: {{criticLevel}} 영화 평론가로서, {{movie}}를 어떻게 생각하시나요?
V2 프롬프트: 당신은 {{criticLevel}} 영화 평론가입니다.

영화 {{movie}}에 대한 상세한 분석을 제공해주세요. 연출, 연기, 스토리, 시각적 효과를 포함하여 평가해주세요.
V1 모델: gpt-4.1-mini
V2 모델: gpt-4.1


### 3.3 라벨 관리

In [19]:
# 특정 라벨로 프롬프트 생성 (같은 이름을 사용하면 새로운 버전으로 생성됨)
langfuse.create_prompt(
    name="movie-critic-chat",
    type="chat",
    prompt=[
        {
            "role": "system",
            "content": "당신은 {{criticLevel}} 영화 평론가입니다. 상세하고 전문적인 분석을 제공해주세요."
        },
        {
            "role": "user",
            "content": "영화 {{movie}}에 대한 평론을 작성해주세요."
        }
    ],
    labels=["staging"],  # staging 환경용
    tags=["movie", "qa", "chat", "detailed"]
)

# 라벨별 프롬프트 가져오기
prompt_production = langfuse.get_prompt("movie-critic-chat", label="production")
prompt_staging = langfuse.get_prompt("movie-critic-chat", label="staging")
prompt_latest = langfuse.get_prompt("movie-critic-chat", label="latest")

In [20]:
# 라벨별 프롬프트 출력
print(f"Production 프롬프트: {prompt_production.prompt}")
print("-" * 100)
print(f"Staging 프롬프트: {prompt_staging.prompt}")
print("-" * 100)
print(f"Latest 프롬프트: {prompt_latest.prompt}")

Production 프롬프트: [{'type': 'message', 'role': 'system', 'content': '당신은 {{criticLevel}} 영화 평론가입니다.'}, {'type': 'message', 'role': 'user', 'content': '영화 {{movie}}를 어떻게 생각하시나요?'}]
----------------------------------------------------------------------------------------------------
Staging 프롬프트: [{'type': 'message', 'role': 'system', 'content': '당신은 {{criticLevel}} 영화 평론가입니다. 상세하고 전문적인 분석을 제공해주세요.'}, {'type': 'message', 'role': 'user', 'content': '영화 {{movie}}에 대한 평론을 작성해주세요.'}]
----------------------------------------------------------------------------------------------------
Latest 프롬프트: [{'type': 'message', 'role': 'system', 'content': '당신은 {{criticLevel}} 영화 평론가입니다. 상세하고 전문적인 분석을 제공해주세요.'}, {'type': 'message', 'role': 'user', 'content': '영화 {{movie}}에 대한 평론을 작성해주세요.'}]


### 3.4 라벨 업데이트

In [21]:
# 기존 프롬프트 버전의 라벨 업데이트
langfuse.update_prompt(
    name="movie-critic-chat",
    version=2,
    new_labels=["production", "v2-stable"]
)

Prompt_Chat(type='chat', prompt=[ChatMessage(role='system', content='당신은 {{criticLevel}} 영화 평론가입니다.', type=None), ChatMessage(role='user', content='영화 {{movie}}를 어떻게 생각하시나요?\n\n<movie_title>\n{{movie_title}}\n</movie_title>', type=None)], name='movie-critic-chat', version=2, config={'model': 'gpt-4.1-mini', 'temperature': 0.7, 'max_tokens': 500}, labels=['production', 'v2-stable'], tags=['movie', 'qa', 'chat', 'detailed'], commit_message='xml tag 적용 ....', resolution_graph=None, id='36bd1756-7805-416b-a4b8-8a1ae722e3d6', createdAt='2026-06-18T12:37:18.119Z', updatedAt='2026-06-18T12:48:50.580Z', projectId='cmqjg29up0075ad0dixyqlcpl', createdBy='cmqjg29tg006yad0d2jenaxo7', isActive=None)

### **[실습 3]**
"movie-critic-chat" 프롬프트를 수정하고, labels 속성은 "staging"으로 지정한 후, staging 버전을 가져와서 내용을 출력하세요.

**단계별 가이드:**
1. `langfuse.create_prompt()`로 같은 이름의 프롬프트 생성 (새 버전 생성)
2. `labels=["staging"]`으로 설정
3. 프롬프트 내용을 수정 (예: 더 상세한 지시사항 추가)
4. `langfuse.get_prompt(name="movie-critic-chat", label="staging")`으로 가져오기
5. 프롬프트 내용 출력 및 확인

**힌트:**
- 같은 이름으로 create하면 자동으로 새 버전 생성
- 라벨을 통해 환경별 프롬프트 관리 가능

In [22]:
# staging 라벨 생성
# 여기에 코드를 작성하세요
# 특정 라벨로 프롬프트 생성 (같은 이름을 사용하면 새로운 버전으로 생성됨)
langfuse.create_prompt(
    name="food_menu_chat",
    type="chat",
    prompt=[
        {
            "role": "system",
            "content": "당신은 {{division}} 메뉴를 골라주는 영양사 입니다."
        },
        {
            "role": "user",
            "content": "{{menu}}에 대한 메뉴 추천해주세요."
        }
    ],
    labels=["staging"],  # staging 환경용
    tags=["food", "dinner", "chat", "detailed"]
)

# staging 라벨 가져오기
# 여기에 코드를 작성하세요
prompt_staging = langfuse.get_prompt("food_menu_chat", label="staging")


---

## 4. LangChain과의 통합

### 4.1 텍스트 프롬프트와 LangChain 통합

In [23]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

# Langfuse 프롬프트를 LangChain과 통합
prompt = langfuse.get_prompt("movie-critic", label="production")

langchain_prompt = PromptTemplate.from_template(
    prompt.get_langchain_prompt(),
    metadata={"langfuse_prompt": prompt},  # Langfuse 자동 링크를 위한 메타데이터
)

# 모델 초기화 (프롬프트 설정 사용)
model = ChatOpenAI(
    model=prompt.config.get("model", "gpt-4.1-mini"),
    temperature=prompt.config.get("temperature", 0.7),
    max_tokens=prompt.config.get("max_tokens", 500)
)

# 체인 생성 및 실행
chain = langchain_prompt | model
response = chain.invoke(
    input={"criticLevel": "전문가", "movie": "인셉션"},
    config={"callbacks": [langfuse_handler]}  # Langfuse 트레이싱을 위한 콜백
)

print(response.content)

물론입니다. 크리스토퍼 놀란 감독의 <인셉션>(Inception, 2010)은 21세기 영화사에서 독보적인 위치를 차지하는 작품입니다. 이 영화는 꿈과 현실의 경계, 무의식의 구조, 인간 심리의 복잡성을 혁신적인 시각효과와 서사 구조로 풀어내며 관객에게 깊은 인상을 남겼습니다. 아래에서 연출, 연기, 스토리, 시각적 효과의 측면에서 <인셉션>을 상세히 분석하겠습니다.

---

### 1. 연출

크리스토퍼 놀란의 연출은 <인셉션>의 핵심입니다. 그는 현실과 꿈의 경계를 복잡하게 교차시키면서도 관객이 영화의 흐름을 따라갈 수 있도록 치밀하게 설계했습니다. 영화는 다층적인 꿈의 구조(드림 레벨)를 통해 시간의 상대성과 인간 심리의 얽힘을 시각적으로 표현합니다. 놀란은 비선형적 내러티브와 플래시백, 빠른 컷 편집을 활용해 혼란스러울 수 있는 이야기를 일관성 있게 풀어냅니다. 특히, 액션 시퀀스와 심리적 긴장감을 동시에 극대화하는 연출은 그의 장기입니다. 다양한 공간과 시간의 레이어가 동시에 진행되는 클라이맥스 시퀀스는 영화사에 남을 명장면 중 하나입니다.

---

### 2. 연기

레오나르도 디카프리오(돔 코브 역)는 죄책감과 상실, 구원에 대한 갈망을 복합적으로 표현하며 영화의 감정적 중심을 이룹니다. 마리옹 꼬띠아르(말 역)는 매혹적이면서도 불안정한 존재감으로 코브의 심리적 갈등을 극적으로 부각시킵니다. 조셉 고든 레빗, 엘렌 페이지(현 엘리엇 페이지), 톰 하디, 킬리언 머피 등 조연 배우들도 각자의 역할을 충실히 소화하며 팀의 케미스트리를 완성합니다. 각각의 캐릭터가 영화의 주제와 구조 속에서 유기적으로 작동합니다.

---

### 3. 스토리

<인셉션>의 스토리는 원래의 꿈(서브젝트의 무의식)에 아이디어를 심는 ‘인셉션’이라는 개념을 중심으로 전개됩니다. 이 아이디어는 단순한 추리/액션물의 플롯을 넘어, 인간의 기억, 죄책감, 꿈과 현실의 구분이라는 철학적 주제를 탐구합니다. 영화는 ‘꿈 안의 꿈’이라는 구조적 장치를 통해 현실과 환상의 경계를 모

### 4.2 챗 프롬프트와 LangChain 통합

In [28]:
from langchain_core.prompts import ChatPromptTemplate

# 챗 프롬프트 통합
chat_prompt = langfuse.get_prompt("movie-critic-chat", label="production", type="chat")

langchain_chat_prompt = ChatPromptTemplate.from_messages(
    chat_prompt.get_langchain_prompt()
)
langchain_chat_prompt.metadata = {"langfuse_prompt": chat_prompt}

# 모델 초기화 (프롬프트 설정 사용)
model = ChatOpenAI(
    model=chat_prompt.config.get("model", "gpt-4.1-mini"),
    temperature=chat_prompt.config.get("temperature", 0.7),
    max_tokens=chat_prompt.config.get("max_tokens", 500)
)

# 체인 실행
chain = langchain_chat_prompt | model
response = chain.invoke(
    input={"criticLevel": "전문가", "movie": "인셉션", "movie_title": "인셉션"},
    config={"callbacks": [langfuse_handler]}
)

print(response.content)

영화 <인셉션>은 크리스토퍼 놀란 감독의 독창적인 작품으로, 복잡하면서도 치밀하게 구성된 이야기와 시각적으로 뛰어난 연출이 돋보이는 영화입니다. 꿈속의 꿈이라는 다층적인 설정을 통해 현실과 환상의 경계를 탐구하며, 관객들에게 깊은 몰입감과 함께 철학적 질문을 던집니다.

특히, 레오나르도 디카프리오의 연기는 캐릭터의 내면적 갈등을 섬세하게 표현해내며, 한스 짐머의 음악과 웅장한 사운드트랙은 긴장감을 극대화합니다. 액션과 심리 스릴러 요소가 절묘하게 조화를 이루어, 단순한 블록버스터를 넘어선 예술적 완성도를 보여준다고 생각합니다.

전반적으로 <인셉션>은 현대 영화 중에서도 보기 드문 지적 자극과 시각적 쾌감을 동시에 제공하는 작품으로, 여러 차례 재관람해도 새로운 해석과 감상을 발견할 수 있는 명작이라고 평가합니다.


### 4.3 플레이스홀더가 있는 프롬프트와 LangChain 통합

In [26]:
from langchain_core.prompts import MessagesPlaceholder

# 플레이스홀더가 있는 프롬프트 가져오기
prompt_with_history = langfuse.get_prompt("movie-critic-with-history", type="chat")

# LangChain 호환 프롬프트로 변환 (미해결 플레이스홀더는 MessagesPlaceholder로 변환)
langchain_prompt_with_placeholder = ChatPromptTemplate.from_messages(
    prompt_with_history.get_langchain_prompt()
)
langchain_prompt_with_placeholder.metadata = {"langfuse_prompt": prompt_with_history}

# 모델 초기화 (프롬프트 설정 사용)
model = ChatOpenAI(
    model=prompt_with_history.config.get("model", "gpt-4.1-mini"),
    temperature=prompt_with_history.config.get("temperature", 0.7),
    max_tokens=prompt_with_history.config.get("max_tokens", 500)
)

# chain 생성
chain = langchain_prompt_with_placeholder | model

# 실행 시 플레이스홀더 값 제공
chat_history = [
    {"role": "user", "content": "안녕하세요! 영화 예산에 대해서 이야기해볼까요?"},
    {"role": "assistant", "content": "안녕하세요! 영화 예산에 대해 어떻게 도와드릴까요?"}
]

response = chain.invoke({
    "criticLevel": "전문가", 
    "movie": "인셉션",
    "chat_history": chat_history
}, config={"callbacks": [langfuse_handler]})  # Langfuse 트레이싱을 위한 콜백

print(response.content)

Placeholders ['chat_history'] have not been resolved. Pass them as keyword arguments to compile().


영화 <인셉션>은 크리스토퍼 놀란 감독의 걸작 중 하나로, 독창적인 스토리와 뛰어난 시각 효과, 그리고 깊이 있는 주제 의식이 돋보이는 작품입니다. 예산 측면에서도 약 1억 6천만 달러(약 1,800억 원) 정도로 알려져 있는데, 이 규모의 예산을 투자해 복잡한 꿈 속 세계를 현실감 있게 구현한 점이 인상적입니다.

특히, 실물 세트를 활용한 촬영과 시각 효과의 조화 덕분에 특수효과가 과하지 않으면서도 몰입감을 높였고, 이는 예산이 단순히 많다고 좋은 영화가 나오는 것이 아니라 효율적이고 창의적인 사용이 중요하다는 것을 보여줍니다. 또한, 톰 하디, 레오나르도 디카프리오 등 스타 배우들의 출연과 음악, 편집 등 모든 요소가 조화를 이루어 큰 성공을 거두었죠.

예산과 결과물의 균형 측면에서 <인셉션>은 영화 산업에서 예산 투자 대비 높은 가치를 창출한 대표적인 사례라고 생각합니다. 혹시 영화 예산과 관련해 더 구체적으로 알고 싶은 부분이 있나요?


### **[실습 4]**
앞에서 정의한 텍스트 기반 프롬프트를 가져와서 LangChain과 통합하여 트레이싱을 실행하고, Langfuse UI에서 결과를 확인하세요.

**단계별 가이드:**
1. `langfuse.get_prompt()`로 텍스트 프롬프트 가져오기
2. `PromptTemplate.from_template()`로 LangChain 프롬프트 생성
3. `metadata`에 Langfuse 프롬프트 링크 추가
4. `ChatOpenAI` 모델 초기화 (config에서 설정 값 가져오기)
5. 체인 생성 및 `invoke()` 실행 (callbacks에 langfuse_handler 추가)
6. Langfuse UI의 Traces 탭에서 실행 결과 확인

**힌트:**
- `prompt.config.get()` 메서드로 안전하게 설정 값 가져오기
- 콜백 핸들러를 추가해야 Langfuse에 트레이싱 기록됨


In [8]:
# 여기에 코드를 추가하세요.
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import MessagesPlaceholder
from langchain_openai import ChatOpenAI

# 플레이스홀더가 있는 프롬프트 가져오기
prompt_with_history = langfuse.get_prompt("movie-critic-with-history", type="chat")

# LangChain 호환 프롬프트로 변환 (미해결 플레이스홀더는 MessagesPlaceholder로 변환)
langchain_prompt_with_placeholder = ChatPromptTemplate.from_messages(
    prompt_with_history.get_langchain_prompt()
)

langchain_prompt_with_placeholder.metadata = {"langfuse_prompt": prompt_with_history}

# 모델 초기화 (프롬프트 설정 사용)
model = ChatOpenAI(
    model=prompt_with_history.config.get("model", "gpt-4.1-mini"),
    temperature=prompt_with_history.config.get("temperature", 0.7),
    max_tokens=prompt_with_history.config.get("max_tokens", 500)
)

# chain 생성
chain = langchain_prompt_with_placeholder | model

# 실행 시 플레이스홀더 값 제공
chat_history = [
    {"role": "user", "content": "안녕하세요! 영화 예산에 대해서 이야기해볼까요?"},
    {"role": "assistant", "content": "안녕하세요! 영화 예산에 대해 어떻게 도와드릴까요?"}
]

response = chain.invoke({
    "criticLevel": "전문가", 
    "movie": "인셉션",
    "chat_history": chat_history
}, config={"callbacks": [langfuse_handler]})  # Langfuse 트레이싱을 위한 콜백

print(response.content)

Placeholders ['chat_history'] have not been resolved. Pass them as keyword arguments to compile().


영화 인셉션(Inception, 2010)은 크리스토퍼 놀란 감독의 대표작 중 하나로, 독창적인 스토리와 뛰어난 시각 효과, 그리고 탄탄한 연출이 돋보이는 작품입니다. 예산 측면에서도 약 1억 6천만 달러(약 1,600억 원) 정도로 꽤 큰 규모였는데, 이 예산이 고도의 특수효과와 세트, 유명 배우들의 캐스팅 등에 효율적으로 사용되었다고 봅니다.

특히 꿈속의 여러 레이어를 시각적으로 구현하는 데 많은 비용과 노력이 들어갔으며, 이는 영화의 몰입감을 크게 높였죠. 예산 대비 높은 수익과 비평적 성공을 거둔 작품으로, 투자 대비 매우 성공적인 영화라고 할 수 있습니다. 혹시 인셉션의 예산과 관련해 더 구체적인 부분이 궁금하신가요?


### **[실습 5]**
앞에서 정의한 chat 기반 프롬프트를 가져와서 LangChain과 통합하여 트레이싱을 실행하고, Langfuse UI에서 결과를 확인하세요.

**단계별 가이드:**
1. `langfuse.get_prompt()`로 챗 프롬프트 가져오기 (`type="chat"` 지정)
2. `ChatPromptTemplate.from_messages()`로 LangChain 챗 프롬프트 생성
3. `metadata`에 Langfuse 프롬프트 링크 추가
4. `ChatOpenAI` 모델 초기화
5. 체인 생성 및 실행 (callbacks 포함)
6. Langfuse UI에서 트레이싱 결과 확인

**힌트:**
- 챗 프롬프트는 `ChatPromptTemplate.from_messages()` 사용
- 실습 4와 동일한 패턴으로 작성

In [ ]:
# 여기에 코드를 추가하세요.